# Soup Lab Light Baseline Profile

One diagnostic follow-up on the original qualified baseline06dc4b1; no new candidate.
Concrete question: were idle periods caused by late CPU submission or already queued CUDA dependencies?
The existing heavy trace has no CUDA sync events and 1.37 million Python events; the same three steps take 2.082 s in phase observation versus 4.223 s under heavy profiling.
Disable shape, stack and memory collection, enable CUDA synchronization activities, and capture steps4-11.
Use separate full64-step phase and timeline children in one T4 session. The first provides a matched timing reference for profiler overhead.
All original task/settings/outputs and runtime source are unchanged. This is not qualification or a performance candidate.


In [ ]:
import subprocess, sys
from pathlib import Path
TARGET = '06dc4b1957fbe1e23b4b32fa420d3205890fa553'
DRIVER_COMMIT = 'bdec46ad36dae645151a1eea84f0f282ad04cf89'
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
DRIVER = Path('/kaggle/working/light-profile-driver')
OUT = Path('/kaggle/working/light-profile')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not DRIVER.exists() and not OUT.exists(), 'Use a fresh session'
run(['git', 'init', str(DRIVER)])
run(['git', '-C', str(DRIVER), 'config', 'core.autocrlf', 'false'])
run(['git', '-C', str(DRIVER), 'config', 'core.eol', 'lf'])
run(['git', '-C', str(DRIVER), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(DRIVER), 'fetch', '--depth', '1', 'origin', DRIVER_COMMIT])
run(['git', '-C', str(DRIVER), 'checkout', '--detach', 'FETCH_HEAD'])
assert subprocess.check_output(['git','-C',str(DRIVER),'rev-parse','HEAD'], text=True).strip() == DRIVER_COMMIT
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True))
run([sys.executable, 'research/test_phase_monitor.py'], cwd=DRIVER)


In [ ]:
# Installation is outside the official timed child processes.
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])


In [ ]:
run([sys.executable, 'research/run_stack_profile.py', '--commit', TARGET, '--out', str(OUT), '--modes', 'phases', 'timeline', '--light-timeline', '--trace-steps', '8'], cwd=DRIVER)


In [ ]:
import json
r=json.loads((OUT/'PROFILE_PROVENANCE.json').read_text())
print('DIAGNOSTIC ONLY:',r['commit'])
print(json.dumps({k:v['exit_code'] for k,v in r['results'].items()},indent=2))
